In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, accuracy_score
import pickle


In [2]:
df = pd.read_csv('Data_Sheet/earthquake_1995-2023.csv')
df.head()


,title,magnitude,date_time,cdi,mmi,alert,tsunami,sig,net,nst,dmin,gap,magType,depth,latitude,longitude,location,continent,country
0,"M 6.5 - 42 km W of Sola, Vanuatu",6.5,16-08-2023 12:47,7,4,green,0,657,us,114,7.177000,25.0,mww,192.955,-13.8814,167.1580,"Sola, Vanuatu",NaN,Vanuatu
1,"M 6.5 - 43 km S of Intipucá, El Salvador",6.5,19-07-2023 00:22,8,6,yellow,0,775,us,92,0.679000,40.0,mww,69.727,12.8140,-88.1265,"Intipucá, El Salvador",NaN,NaN
2,"M 6.6 - 25 km ESE of Loncopué, Argentina",6.6,17-07-2023 03:05,7,5,green,0,899,us,70,1.634000,28.0,mww,171.371,-38.1911,-70.3731,"Loncopué, Argentina",South America,Argentina
3,"M 7.2 - 98 km S of Sand Point, Alaska",7.2,16-07-2023 06:48,6,6,green,1,860,us,173,0.907000,36.0,mww,32.571,54.3844,-160.6990,"Sand Point, Alaska",NaN,NaN
4,M 7.3 - Alaska Peninsula,7.3,16-07-2023 06:48,0,5,NaN,1,820,at,79,0.879451,172.8,Mi,21.000,54.4900,-160.7960,Alaska Peninsula,NaN,NaN


In [3]:
#Null values 

df.isna().sum()

title          0
magnitude      0
date_time      0
cdi            0
mmi            0
alert        551
tsunami        0
sig            0
net            0
nst            0
dmin           0
gap            0
magType        0
depth          0
latitude       0
longitude      0
location       6
continent    716
country      349
dtype: int64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 19 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      1000 non-null   object 
 1   magnitude  1000 non-null   float64
 2   date_time  1000 non-null   object 
 3   cdi        1000 non-null   int64  
 4   mmi        1000 non-null   int64  
 5   alert      449 non-null    object 
 6   tsunami    1000 non-null   int64  
 7   sig        1000 non-null   int64  
 8   net        1000 non-null   object 
 9   nst        1000 non-null   int64  
 10  dmin       1000 non-null   float64
 11  gap        1000 non-null   float64
 12  magType    1000 non-null   object 
 13  depth      1000 non-null   float64
 14  latitude   1000 non-null   float64
 15  longitude  1000 non-null   float64
 16  location   994 non-null    object 
 17  continent  284 non-null    object 
 18  country    651 non-null    object 
dtypes: float64(6), int64(5), object(8)
memory usage: 

In [5]:
#Fillup the null values
df['alert'] = df['alert'].fillna(df['alert'].mode()[0])
df["location"] = df["location"].fillna("Unknown")
df['location'] = df['location'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['continent'] = df['continent'].fillna('Unknown')

df['dmin'] = round(df['dmin'],3)
df['depth'] = round(df['depth'],2)

In [6]:
#Do saperate the Date and time values.

df['date_time'] = pd.to_datetime(df['date_time'], errors='coerce')

df['year'] = df['date_time'].dt.year
df['month'] = df['date_time'].dt.month
df['day'] = df['date_time'].dt.day
df['hour'] = df['date_time'].dt.hour
df['minute'] = df['date_time'].dt.minute

df.drop(columns=['date_time'], inplace=True)


C:\Users\Krushn\AppData\Local\Temp\ipykernel_8112\4156414255.py:3: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['date_time'] = pd.to_datetime(df['date_time'], errors='coerce')


In [7]:
df.to_csv("Data_Sheet/non_scaled_cleaned_data.csv")

In [8]:
categorical_columns = ['title','alert','net','magType','continent','country','location','tsunami']

encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

In [9]:
features = [
    'title',    'cdi',    'mmi',    'net',    'nst',    'dmin',
    'gap',    'magType',    'depth',    'latitude',    'longitude',
    'location',    'continent',    'country',    'year',    'month',
    'day',    'hour',    'minute' , 'tsunami'
]

X = df[features]

In [10]:
print("Training Magnitude Prediction Model...\n")

# Features and Target
X_magnitude = X
y_magnitude = df["magnitude"]

# Train Test Split
X_train_magnitude, X_test_magnitude, y_train_magnitude, y_test_magnitude = train_test_split(
    X_magnitude, y_magnitude,
    test_size=0.2,
    random_state=42
)

# Model
magnitude_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train
magnitude_model.fit(X_train_magnitude, y_train_magnitude)

# Prediction
y_pred_magnitude = magnitude_model.predict(X_test_magnitude)

# Evaluation
score_magnitude = r2_score(y_test_magnitude, y_pred_magnitude)

print(f"R² Score : {score_magnitude}")

Training Magnitude Prediction Model...

R² Score : 0.9987950345417066


In [11]:
print("Training Significance Prediction Model...\n")

# Features and Target
X_sig = X
y_sig = df['sig']

# Train Test Split
X_train_sig, X_test_sig, y_train_sig, y_test_sig = train_test_split(
    X_sig, y_sig,
    test_size=0.2,
    random_state=42
)

# Model
sig_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train
sig_model.fit(X_train_sig, y_train_sig)

# Prediction
y_pred = sig_model.predict(X_test_sig)

# Evaluation
score_sig = r2_score(y_test_sig, y_pred)

print(f"R² Score : {score_sig}")

Training Significance Prediction Model...

R² Score : 0.6094792941332885


In [12]:
print("Training Alert Prediction Model...\n")

# Features and Target
X_alert = X
y_alert = df['alert']

# Train Test Split
X_train_alert, X_test_alert, y_train_alert, y_test_alert = train_test_split(
    X_alert, y_alert,
    test_size=0.2,
    random_state=42
)

# Model
alert_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train
alert_model.fit(X_train_alert, y_train_alert)

# Prediction
y_pred_alert = alert_model.predict(X_test_alert)

# Evaluation
score_alert = accuracy_score(y_test_alert, y_pred_alert)

print(f"Accuracy : {score_alert}")

Training Alert Prediction Model...

Accuracy : 0.92


In [13]:
print("Training Tsunami Prediction Model...\n")

# Features and Target
X_tsunami = X
y_tsunami = df['tsunami']

# Train Test Split
X_train_tsunami, X_test_tsunami, y_train_tsunami, y_test_tsunami = train_test_split(
    X_tsunami, y_tsunami,
    test_size=0.2,
    random_state=42
)

# Model
tsunami_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train
tsunami_model.fit(X_train_tsunami, y_train_tsunami)

# Prediction
y_pred_tsunami = tsunami_model.predict(X_test_tsunami)

# Evaluation
score_tsunami = accuracy_score(y_test_tsunami, y_pred_tsunami)

print(f"Accuracy : {score_tsunami}")

Training Tsunami Prediction Model...

Accuracy : 1.0


In [14]:
models = {
    "magnitude_model": magnitude_model,
    "significance_model": sig_model,
    "alert_model": alert_model,
    "tsunami_model": tsunami_model,
    "encoders": encoders
}

with open("Models/earthquake_models.pkl", "wb") as file:
    pickle.dump(models, file)

print("Model is created into the 'Models/' folder...")

Model is created into the 'Models/' folder...


In [15]:
# User Inputs

title_value = "M 7.8 - Central Turkey"

continent_value = "Asia"
country_value = "Turkiye"
location_value = "Central Turkey"

year = 2023
month = 2
day = 6
hour = 1
minute = 17

latitude = 37.2012
longitude = 36.9997

depth = 17.925

magType_value = "mww"

cdi = 9
mmi = 9

net_value = "us"

nst = 118

dmin = 1.919

gap = 32
nst = 168

dmin = 0.345

gap = 24.8

tsunami_value = 0  # 0 = No, 1 = Yes

print("User input completed....")

User input completed....


In [16]:
# Encode the categorical columns using the fitted encoders from training

title = encoders["title"].transform([title_value])[0]
continent = encoders["continent"].transform([continent_value])[0]
country = encoders["country"].transform([country_value])[0]
location = encoders["location"].transform([location_value])[0]
magType = encoders["magType"].transform([magType_value])[0]
net = encoders["net"].transform([net_value])[0]
tsunami = encoders["tsunami"].transform([str(tsunami_value)])[0]

print("User input Encoding is completed.")

User input Encoding is completed.


In [17]:
# Dataframe of the user input

user_data = pd.DataFrame({
    "title":[title],
    "cdi":[cdi],
    "mmi":[mmi],
    "net":[net],
    "nst":[nst],
    "dmin":[dmin],
    "gap":[gap],
    "magType":[magType],
    "depth":[depth],
    "latitude":[latitude],
    "longitude":[longitude],
    "location":[location],
    "continent":[continent],
    "country":[country],
    "year":[year],
    "month":[month],
    "day":[day],
    "hour":[hour],
    "minute":[minute],
    "tsunami":[tsunami]
})

#Prediction of multiple models

predicted_magnitude = magnitude_model.predict(user_data)[0]
predicted_sig = sig_model.predict(user_data)[0]
predicted_alert = alert_model.predict(user_data)[0]
predicted_tsunami = tsunami_model.predict(user_data)[0]

predicted_alert = encoders["alert"].inverse_transform([predicted_alert])[0]

predicted_tsunami = encoders["tsunami"].inverse_transform([predicted_tsunami])[0]

In [18]:
print("\n" + "=" * 50)
print("      EARTHQUAKE PREDICTION REPORT")
print("=" * 50)

print(f"Predicted Magnitude     : {predicted_magnitude:.2f}")
print(f"Predicted Significance  : {predicted_sig:.2f}")
print(f"Predicted Alert Level   : {predicted_alert}")
print(f"Predicted Tsunami       : {'Yes' if predicted_tsunami == 1 else 'No'}")

print("=" * 50)


      EARTHQUAKE PREDICTION REPORT
Predicted Magnitude     : 7.80
Predicted Significance  : 2198.05
Predicted Alert Level   : red
Predicted Tsunami       : No


In [19]:
# Earthquake Safety Recommendation System

print("\nGenerating Safety Recommendations...\n")

recommendations = []

# Magnitude Based Recommendation

if predicted_magnitude < 3:
    recommendations.append(
        "Very Minor Earthquake. Usually not felt. No immediate danger.")

elif predicted_magnitude < 5:
    recommendations.append(
        "Light Earthquake. Stay alert and avoid panic.")

elif predicted_magnitude < 6:
    recommendations.append(
        "Moderate Earthquake. Inspect buildings for damage and avoid weak structures.")

elif predicted_magnitude < 7:
    recommendations.append(
        "Strong Earthquake. Stay in an open area and prepare for aftershocks.")

elif predicted_magnitude < 8:
    recommendations.append(
        "Major Earthquake. Emergency response is highly recommended.")

else:
    recommendations.append(
        "Great Earthquake. High risk of severe destruction. Immediate evacuation is recommended.")

# Depth Recommendation

if depth < 70:
    recommendations.append(
        "Shallow earthquake detected. Surface damage can be significant.")

elif depth < 300:
    recommendations.append(
        "Intermediate-depth earthquake. Damage depends on local conditions.")

else:
    recommendations.append(
        "Deep earthquake. Surface damage is usually less severe.")
    

# Significance Recommendation

if predicted_sig < 200:
    recommendations.append(
        "Low significance event.")

elif predicted_sig < 500:
    recommendations.append(
        "Moderate significance. Stay informed through official updates.")

elif predicted_sig < 800:
    recommendations.append(
        "High significance. Emergency services may be activated.")

else:
    recommendations.append(
        "Extremely significant earthquake. Follow disaster management instructions immediately.")
    

# Alert Recommendation

if predicted_alert == "green":
    recommendations.append(
        "Green Alert: Minimal impact expected.")

elif predicted_alert == "yellow":
    recommendations.append(
        "Yellow Alert: Some local damage possible.")

elif predicted_alert == "orange":
    recommendations.append(
        "Orange Alert: Serious damage expected. Keep emergency kits ready.")

elif predicted_alert == "red":
    recommendations.append(
        "RED ALERT: Major destruction possible. Evacuate dangerous buildings immediately.")
    
# Tsunami Recommendation

if predicted_tsunami == "1":
    recommendations.append(
        "TSUNAMI WARNING: Move immediately to higher ground.")
    recommendations.append(
        "Stay away from beaches, rivers and coastal regions.")
    recommendations.append(
        "Wait for official clearance before returning.")

else:
    recommendations.append(
        "No tsunami threat predicted.")

# Gap Recommendation

if gap > 180:
    recommendations.append(
        "Large station gap. Prediction confidence may be lower.")

# Number of Stations

if nst < 20:
    recommendations.append(
        "Limited seismic stations detected. Prediction reliability may decrease."
    )

elif nst > 100:
    recommendations.append(
        "Large number of seismic stations contributed. Prediction confidence is higher."
    )

# Population Awareness

if cdi >= 7:
    recommendations.append(
        "Many people are expected to strongly feel this earthquake."
    )

if mmi >= 7:
    recommendations.append(
        "Severe shaking expected. Structural damage is possible."
    )

# General Emergency Tips

recommendations.append(
    "Keep emergency contact numbers available."
)

recommendations.append(
    "Carry a first-aid kit and drinking water."
)

recommendations.append(
    "Stay away from damaged buildings."
)

recommendations.append(
    "Expect aftershocks and remain cautious."
)

recommendations.append(
    "Follow official government and disaster management instructions."
)

# Print Recommendations

print("="*65)
print("        EARTHQUAKE SAFETY RECOMMENDATIONS")
print("="*65)

for i, rec in enumerate(recommendations, start=1):
    print(f"{i}. {rec}")

print("="*65)

# Overall Risk Level

risk_score = 0

if predicted_magnitude >= 7:
    risk_score += 3
elif predicted_magnitude >= 6:
    risk_score += 2
elif predicted_magnitude >= 5:
    risk_score += 1

if predicted_sig >= 700:
    risk_score += 2

if predicted_alert == "red":
    risk_score += 3
elif predicted_alert == "orange":
    risk_score += 2
elif predicted_alert == "yellow":
    risk_score += 1
elif predicted_alert == 'green':
    risk_score +=0 

if depth < 70:
    risk_score += 1

# Final Risk

if risk_score <= 2:
    risk = "LOW"

elif risk_score <= 5:
    risk = "MODERATE"
elif risk_score <= 8:
    risk = "HIGH"

else:
    risk = "EXTREME"

print("\nOverall Earthquake Risk Level :", risk)


Generating Safety Recommendations...

        EARTHQUAKE SAFETY RECOMMENDATIONS
1. Major Earthquake. Emergency response is highly recommended.
2. Shallow earthquake detected. Surface damage can be significant.
3. Extremely significant earthquake. Follow disaster management instructions immediately.
4. RED ALERT: Major destruction possible. Evacuate dangerous buildings immediately.
5. No tsunami threat predicted.
6. Large number of seismic stations contributed. Prediction confidence is higher.
7. Many people are expected to strongly feel this earthquake.
8. Severe shaking expected. Structural damage is possible.
9. Keep emergency contact numbers available.
10. Carry a first-aid kit and drinking water.
11. Stay away from damaged buildings.
12. Expect aftershocks and remain cautious.
13. Follow official government and disaster management instructions.

Overall Earthquake Risk Level : EXTREME


In [22]:
#Saved last scaled data Files 

df.to_csv("Data_Sheet/cleande_and_encodeed_dara.csv")